In [1]:
import os
import psycopg2
import pandas as pd
from urllib.request import urlopen
import json
import math

In [2]:
# environment variable can take two values: DEV (development), PRD (production)
env = 'DEV'

# if set to true every existing entry will be deleted
clean = False

In [3]:
# for production we will use following database uri variable
DATABASE_URI = ''

In [5]:
#iterate through list of tradble stocks
try:
    # check environment
    if env == 'DEV':
        con = psycopg2.connect(host="localhost", user='postgres', database='value-investing-dev', port='5432', password='v,1846PSVv,1846PSV')
    elif env == 'PRD':
        con = psycopg2.connect(DATABASE_URI)

    #create cursor to execute sql statements
    cur = con.cursor()

    # first we will check if already entries are present in table; if yes we will delete all of them
    sql_select_query = """select * from public.fmp_companylogos"""

    # execute the sql query
    cur.execute(sql_select_query)

    ratios = cur.fetchall()

    if len(ratios) > 0 and clean:
        sql_delete_query = """delete from public.fmp_companylogos"""

        #delete all entries
        cur.execute(sql_delete_query)

        #commit deletion
        con.commit()

    # API_KEY = os.environ["FMP_API_KEY"]
    API_KEY = "111db6ad2ca657d8b3a17d356a9a1a71"
    # allocate memory for response
    api_response = []

    # get symbols list that are available on FMP
    symbols = f'https://financialmodelingprep.com/api/v3/stock/list?apikey={API_KEY}'
    response = urlopen(symbols)
    symbols = response.read().decode("utf-8")
    symbols = pd.DataFrame(json.loads(symbols))

    # only gets symbols of type stock
    symbols = symbols[symbols['type'] == 'stock']

    rows_nr = symbols.shape[0]

    # iterate through symbols and prepare response
    for index, row in symbols.iterrows():
        print(f'progress: {round(index/rows_nr*100, 2)} %, index: {index}')

        if index > 18884:

            #first we will check if symbol already exists
            sql_select_query = """select * from public.fmp_companylogos where symbol = %s"""

            cur.execute(sql_select_query, (row["symbol"],))

            #fetch first entry
            db_row = cur.fetchone()

            # if ticker is not yet present we will create a new entry in db
            if db_row is None:
                # check if company profile is available
                profile = f'https://financialmodelingprep.com/api/v3/profile/{row["symbol"]}?apikey={API_KEY}'
                
                response_profile = urlopen(profile)
                profile = response_profile.read().decode("utf-8")
                profile = pd.DataFrame(json.loads(profile))

                sql_insert_query = """INSERT INTO public.fmp_companylogos(SYMBOL, "imageURL") VALUES(%s, %s)"""
                cur.execute(sql_insert_query, (row['symbol'], profile['image'][0]))
                con.commit()
            else:
                print(f'ticker is present: {row["symbol"]}')

    cur.close()

except Exception as e:
    print('Error: ', e)


progress: 0.0 %, index: 0
progress: 0.0 %, index: 1
progress: 0.0 %, index: 2
progress: 0.01 %, index: 3
progress: 0.01 %, index: 4
progress: 0.01 %, index: 5
progress: 0.01 %, index: 6
progress: 0.01 %, index: 7
progress: 0.01 %, index: 8
progress: 0.02 %, index: 9
progress: 0.02 %, index: 10
progress: 0.02 %, index: 11
progress: 0.02 %, index: 12
progress: 0.03 %, index: 14
progress: 0.03 %, index: 15
progress: 0.03 %, index: 16
progress: 0.03 %, index: 17
progress: 0.03 %, index: 18
progress: 0.04 %, index: 19
progress: 0.04 %, index: 20
progress: 0.04 %, index: 21
progress: 0.04 %, index: 22
progress: 0.04 %, index: 23
progress: 0.04 %, index: 24
progress: 0.05 %, index: 25
progress: 0.05 %, index: 26
progress: 0.05 %, index: 27
progress: 0.05 %, index: 28
progress: 0.05 %, index: 29
progress: 0.06 %, index: 30
progress: 0.06 %, index: 31
progress: 0.06 %, index: 32
progress: 0.06 %, index: 33
progress: 0.06 %, index: 34
progress: 0.07 %, index: 35
progress: 0.07 %, index: 36
progr